TEST code

In [1]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import sounddevice as sd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
SAMPLE_RATE = 16000
FRAME_LENGTH = int(0.025 * SAMPLE_RATE)   # 400
HOP_LENGTH = int(0.010 * SAMPLE_RATE)     # 160
N_FFT = 512
N_MFCC = 13

In [4]:
def extract_features(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

        # Skip very short audio
        if len(y) < 0.1 * sr:
            return None

        # Pre-emphasis
        y = np.append(y[0], y[1:] - 0.97 * y[:-1])

        # MFCC
        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=N_MFCC,
            n_fft=N_FFT,
            win_length=FRAME_LENGTH,
            hop_length=HOP_LENGTH,
            window="hamming"
        )

        # Delta MFCC (safe width)
        mfcc_delta = librosa.feature.delta(mfcc, width=3)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2, width=3)

        # ZCR
        zcr = librosa.feature.zero_crossing_rate(
            y,
            frame_length=FRAME_LENGTH,
            hop_length=HOP_LENGTH
        )

        # Stack all features
        features = np.vstack([mfcc, mfcc_delta, mfcc_delta2, zcr])

        # Temporal pooling
        feature_vector = np.hstack([
            np.mean(features, axis=1),
            np.std(features, axis=1)
        ])

        return feature_vector

    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None


In [5]:
class ELM:
    def __init__(self, input_size, hidden_size=300):
        self.W = np.random.randn(input_size, hidden_size)
        self.b = np.random.randn(hidden_size)

    def _sigmoid(self, x):
        x = np.clip(x, -50, 50)   # prevent overflow
        return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        H = self._sigmoid(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._sigmoid(X @ self.W + self.b)
        return H @ self.beta

    def predict_class(self, X, threshold=0.5):
        return (self.predict(X) >= threshold).astype(int)

In [6]:
def train_model(csv_path="dataset_labels.csv"):
    print("Loading dataset CSV...")
    df = pd.read_csv(csv_path)

    X, y = [], []

    for _, row in df.iterrows():
        features = extract_features(row["filepath"])
        if features is not None:
            X.append(features)
            y.append(row["label"])

    X = np.array(X)
    y = np.array(y)

    print("Feature matrix:", X.shape)
    print("Labels:", y.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    elm = ELM(input_size=X.shape[1], hidden_size=300)
    elm.fit(X_train, y_train)

    print("ELM training completed")

    y_pred = elm.predict_class(X_test, threshold=0.5)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    # Save model
    with open("elm_model.pkl", "wb") as f:
        pickle.dump(elm, f)

    print("Model saved as elm_model.pkl")

In [7]:
def load_model():
    with open("elm_model.pkl", "rb") as f:
        return pickle.load(f)

In [8]:
def continuous_detection(elm, threshold=0.4):
    print("🎧 Continuous Siren Detection Started (Ctrl+C to stop)")
    print("Threshold:", threshold)

    while True:
        try:
            # Record 1 second audio
            audio = sd.rec(
                int(SAMPLE_RATE),
                samplerate=SAMPLE_RATE,
                channels=1,
                dtype="float32"
            )
            sd.wait()

            audio = audio.flatten()
            sf.write("temp_audio.wav", audio, SAMPLE_RATE)

            feature = extract_features("temp_audio.wav")

            if feature is None:
                print("Invalid audio")
                continue

            feature = feature.reshape(1, -1)
            result = elm.predict_class(feature, threshold=threshold)

            if result[0] == 1:
                print("🚑 SIREN DETECTED")
            else:
                print("No siren")

            time.sleep(0.1)

        except KeyboardInterrupt:
            print("\nDetection stopped")
            break


In [9]:
if __name__ == "__main__":

    # STEP 1: Train once (comment this after model is saved)
    # train_model("dataset_labels.csv")

    # STEP 2: Load trained model
    elm_model = load_model()

    # STEP 3: Start continuous detection
    continuous_detection(elm_model, threshold=0.4)

FileNotFoundError: [Errno 2] No such file or directory: 'elm_model.pkl'